In [9]:
from pyspark.sql import SparkSession
from transforms.utils import LOCAL_SPARK, get_current_calmonth

spark = (
    SparkSession.builder.remote(LOCAL_SPARK).getOrCreate() # type: ignore
)

_, year, month, day = get_current_calmonth()

bronze_path = f"ofs://om/lakehouse/bronze/breweries/usa/{year}/{month}/{day}/breweries.json"
df = spark.read.json(bronze_path)
df.show()
# df.printSchema()

+--------------------+---------+---------+------------+--------------+-------------+--------------------+---------------+----------------+--------------------+----------+-----------+-------------+--------------+--------------------+--------------------+
|           address_1|address_2|address_3|brewery_type|          city|      country|                  id|       latitude|       longitude|                name|     phone|postal_code|        state|state_province|              street|         website_url|
+--------------------+---------+---------+------------+--------------+-------------+--------------------+---------------+----------------+--------------------+----------+-----------+-------------+--------------+--------------------+--------------------+
|      1716 Topeka St|     NULL|     NULL|       micro|        Norman|United States|5128df48-79fc-4f0...|    35.25738891|    -97.46818222|    (405) Brewing Co|4058160490| 73069-8224|     Oklahoma|      Oklahoma|      1716 Topeka St|http:/

In [7]:
silver_breweries = (
    spark.read.format("iceberg")
    .load("lakehouse.silver.breweries")
)
silver_breweries.show()

+--------------------+--------------------+------------+--------------------+---------+---------+--------------------+-------------------+-----------+-------+----------+----------+----------------+--------------------+--------------------+
|          brewery_id|        brewery_name|brewery_type|           address_1|address_2|address_3|                city|     state_province|postal_code|country| longitude|  latitude|           phone|         website_url|                ldts|
+--------------------+--------------------+------------+--------------------+---------+---------+--------------------+-------------------+-----------+-------+----------+----------+----------------+--------------------+--------------------+
|4feeca61-265e-419...|             Airbräu|     brewpub|München Airport C...|     NULL|     NULL|            Oberding|             Bayern|      85356|Germany|11.7873402|48.3540889| +49 89 97593111|http://www.airbra...|2026-07-08 05:41:...|
|5b99dc11-165e-4cf...|          Adler-Br

In [13]:
from pyspark.sql.functions import col
usa_sample = (
    silver_breweries
    .filter(col("country") == "United States")
    .limit(10)
)

ger_sample = (
    silver_breweries
    .filter(col("country") == "Germany")
    .limit(10)
)

usa_sample.show()
ger_sample.show()

+--------------------+--------------------+------------+--------------------+---------+---------+-----------+--------------+-----------+-------------+------------+-----------+----------+--------------------+--------------------+
|          brewery_id|        brewery_name|brewery_type|           address_1|address_2|address_3|       city|state_province|postal_code|      country|   longitude|   latitude|     phone|         website_url|                ldts|
+--------------------+--------------------+------------+--------------------+---------+---------+-----------+--------------+-----------+-------------+------------+-----------+----------+--------------------+--------------------+
|6d14b220-8926-452...|10 Barrel Brewing Co|       large|       62970 18th St|     NULL|     NULL|       Bend|        Oregon| 97701-9847|United States| -121.281706|44.08683531|5415851007|http://www.10barr...|2026-07-08 05:41:...|
|4ffda196-dd59-44a...| 105 West Brewing Co|       micro|        1043 Park St|     NU